# AutoClips on Kaggle (GPU, batch + optional Telegram Approve/Deny)

Renders viral clips from a YouTube video into `/kaggle/working`. Cell **8b** sends them to Telegram — with the repo's **Approve/Deny buttons** when `USE_APPROVAL_BUTTONS=True` — and cell **8c** starts the repo's **polling listener** (outbound long-polling, Kaggle-OK, no inbound ports) so button taps actually run. No FastAPI server is started.

One-time notebook setup (right sidebar):
1. **Accelerator → GPU T4 x2** (P100 was retired Sep 2026).
2. **Internet → ON** (needed for pip, yt-dlp, Gemini, Telegram).
3. **Persistence → Files only** (so `/kaggle/working` survives session restarts).

Run cells top-to-bottom. Limits: ~12h per session, ~30h GPU/week, ~20GB in `/kaggle/working`, 60-min idle timeout.

All sending/approval code is reused from the repo (`app/telegram/sending.py`, `handlers_buttons.py`, `services/approvals.py`) — the notebook only wires startup without `uvicorn`.


## 0. Environment check (GPU + ffmpeg)

If ffmpeg/ffprobe are missing, the next cell tries to install them. If GPU shows 0 devices, stop: Settings → Accelerator → GPU T4 x2.

In [ ]:
import shutil, sys
print('python:', sys.version.split()[0])
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>&1 | head -5
print('ffmpeg :', shutil.which('ffmpeg') or 'MISSING')
print('ffprobe:', shutil.which('ffprobe') or 'MISSING')

try:
    import torch
    print('torch:', torch.__version__, '| cuda_available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu:', torch.cuda.get_device_name(0))
except Exception as e:
    print('torch not importable yet (will come with requirements):', e)

## 1. Get the project code under `/kaggle/working`

Option A (easiest): upload `autoclips-python.zip` via **Add input → Upload** or the file browser, copying it to `/kaggle/working/`, then run the unzip cell.
Option B: `git clone` your repo (uncomment the last line).

In [ ]:
import glob, os

WORK = '/kaggle/working/autoclips-python'
os.makedirs('/kaggle/working', exist_ok=True)
print('WORK =', WORK)
print('zips in /kaggle/working:', sorted(glob.glob('/kaggle/working/*.zip')))
print('WORK exists:', os.path.isdir(WORK), '| cwd:', os.getcwd())

In [ ]:
import glob, os, zipfile

WORK = '/kaggle/working/autoclips-python'
zips = sorted(glob.glob('/kaggle/working/*.zip'))
if zips and not os.path.isdir(os.path.join(WORK, 'app')):
    z = zips[-1]
    print('Extracting', z)
    with zipfile.ZipFile(z) as zf:
        zf.extractall('/kaggle/working/')
    # Normalize a single top-level folder to WORK
    if not os.path.isdir(os.path.join(WORK, 'app')):
        folders = [d for d in os.listdir('/kaggle/working')
                   if os.path.isdir(f'/kaggle/working/{d}') and d != 'autoclips-python']
        # find whichever folder now contains app/
        for d in folders:
            if os.path.isdir(f'/kaggle/working/{d}/app'):
                import shutil
                shutil.rmtree(WORK, ignore_errors=True)
                os.rename(f'/kaggle/working/{d}', WORK)
                break
    print('extracted.')
elif os.path.isdir(os.path.join(WORK, 'app')):
    print('Project already at', WORK)
else:
    print('No zip found and no project dir. Upload autoclips-python.zip to /kaggle/working/ first,')
    print('or use OPTION B below.')
print(sorted(os.listdir(WORK)) if os.path.isdir(WORK) else 'WORK missing')

# OPTION B (instead of zip): uncomment and set your repo URL
# !git clone https://github.com/YOUR_USER/autoclips-python.git /kaggle/working/autoclips-python

## 2. System deps (ffmpeg only if missing)

Kaggle images usually ship ffmpeg. This cell is a no-op when present.

In [ ]:
import shutil
if shutil.which('ffmpeg') and shutil.which('ffprobe'):
    print('ffmpeg + ffprobe already present — skipping apt.')
else:
    print('ffmpeg missing — attempting install (may need sudo on Kaggle)...')
    import subprocess
    # Try without sudo first, then with sudo.
    r = subprocess.run('apt-get update -qq && apt-get install -y -qq ffmpeg fonts-dejavu-core 2>&1 | tail -3', shell=True, capture_output=True, text=True)
    print(r.stdout[-1500:] or r.stderr[-1500:])
    if not shutil.which('ffmpeg'):
        r2 = subprocess.run('sudo apt-get update -qq && sudo apt-get install -y -qq ffmpeg fonts-dejavu-core 2>&1 | tail -3', shell=True, capture_output=True, text=True)
        print(r2.stdout[-1500:] or r2.stderr[-1500:])
    print('ffmpeg :', shutil.which('ffmpeg') or 'STILL MISSING — check notebook logs')
    print('ffprobe:', shutil.which('ffprobe') or 'STILL MISSING')
!ffmpeg -version 2>&1 | head -1
# Devanagari (Hindi) captions need a Devanagari font — DejaVu alone renders them blank.
import subprocess as _sp
_has_deva = False
try:
    _o = _sp.run(['fc-list', ':lang=hi', 'family'], capture_output=True, text=True, timeout=20)
    _has_deva = bool((_o.stdout or '').strip())
except Exception:
    pass
print('devanagari font:', 'present ✓' if _has_deva else 'MISSING — installing fonts-noto-core...')
if not _has_deva:
    _r = _sp.run('apt-get install -y -qq fonts-noto-core 2>&1 | tail -2 || sudo apt-get install -y -qq fonts-noto-core 2>&1 | tail -2', shell=True, capture_output=True, text=True)
    print((_r.stdout or _r.stderr or '')[-800:])
    try:
        _o2 = _sp.run(['fc-list', ':lang=hi', 'family'], capture_output=True, text=True, timeout=20)
        print('devanagari font now:', 'present ✓' if (_o2.stdout or '').strip() else 'STILL MISSING — Hindi subs will render blank')
    except Exception as _e:
        print('font re-check failed:', _e)

# yt-dlp needs a JS runtime for YouTube DASH formats (else ".part rename" failures).
# The downloader passes --js-runtimes automatically; just ensure node or deno exists.
import shutil as _sh
_rt = _sh.which("deno") or _sh.which("node") or _sh.which("nodejs")
print("js runtime:", _rt or "MISSING")
if not _rt:
    print("installing nodejs for yt-dlp ...")
    import subprocess as _sp2
    _r = _sp2.run("apt-get install -y -qq nodejs 2>&1 | tail -2 || sudo apt-get install -y -qq nodejs 2>&1 | tail -2", shell=True, capture_output=True, text=True)
    print((_r.stdout or _r.stderr or "")[-800:])
    print("js runtime now:", _sh.which("deno") or _sh.which("node") or _sh.which("nodejs") or "STILL MISSING")


## 3. Python deps (reuse Kaggle's CUDA torch)

Same rule as Colab: do NOT force a torch reinstall. First install takes a few minutes; later sessions reuse `/kaggle/working` caches where possible.

In [ ]:
%cd /kaggle/working/autoclips-python
!pip -q install -r requirements.txt
import torch
print('torch', torch.__version__, '| cuda_available:', torch.cuda.is_available())
import ctranslate2
n = ctranslate2.get_cuda_device_count()
print('CTranslate2 CUDA devices:', n)
if n == 0:
    print('WARNING: no GPU visible. Set Settings → Accelerator → GPU T4 x2, then restart the session.')

## 4. Configure — EDIT THIS CELL, then run

Paste-in-cell secrets (nothing is saved outside `/kaggle/working`). `cookies.txt` is optional — only needed if YouTube blocks Kaggle IPs (`Sign in to confirm you're not a bot`); upload it to `/kaggle/working/` and it is picked up automatically.

In [ ]:
import os

WORK = '/kaggle/working/autoclips-python'
CONFIG_DIR = '/kaggle/working/autoclips-config'
os.makedirs(CONFIG_DIR, exist_ok=True)
os.chdir(WORK)

VIDEO_URL = 'https://www.youtube.com/watch?v=VIDEO_ID'
GEMINI_API_KEY = 'your-gemini-key'          # required for analyze/viral
TELEGRAM_BOT_TOKEN = ''                    # required only if SEND_TO_TELEGRAM=True below
TELEGRAM_CHAT_ID = ''                      # your numeric chat id (message @userinfobot to find yours)
SEND_TO_TELEGRAM = False                   # True = send every rendered clip after cell 8
USE_APPROVAL_BUTTONS = True                # True = repo's send_clip_to_telegram with Approve/Deny buttons (needs poller cell 8c for clicks)
START_TELEGRAM_POLLER = True               # True = cell 8c starts polling so Approve/Deny clicks work (outbound, Kaggle-OK)
MAX_CLIPS = None                           # e.g. 5 to cap, None = let Gemini decide
SUBTITLES_MODE = 'hinglish'                # none | english | native | hinglish (Latin-script Hindi)
VERTICAL_CROP = True                       # 9:16 face-aware crop
WHISPER_MODEL_SIZE = 'large-v3-turbo'      # small | medium | large-v3-turbo (T4 sweet spot)
FFMPEG_PRESET = 'veryfast'
RENDER_CRF = '20'
FACE_MODEL = 'yolo'
YT_CLIENT_ID = ''
YT_CLIENT_SECRET = ''
YT_REFRESH_TOKEN = ''                       # via scripts/get_youtube_token.py (local once), paste here
META_PAGE_TOKEN = ''
FB_PAGE_ID = ''
IG_USER_ID = ''
USE_LOCAL_VIDEO = False
LOCAL_VIDEO_PATH = '/kaggle/working/my_video.mp4'  # upload first, then set True

cookies = '/kaggle/working/cookies.txt' if os.path.exists('/kaggle/working/cookies.txt') else ''
if cookies:
    print('Using YouTube cookies:', cookies)

env = f'''TELEGRAM_BOT_TOKEN={TELEGRAM_BOT_TOKEN}
TELEGRAM_CHAT_ID={TELEGRAM_CHAT_ID}
GEMINI_API_KEY={GEMINI_API_KEY}
WHISPER_MODEL_SIZE={WHISPER_MODEL_SIZE}
WHISPER_DEVICE=cuda
WHISPER_COMPUTE_TYPE=float16
WHISPER_VAD_FILTER=true
WHISPER_MIN_SILENCE_MS=400
WHISPER_SPEECH_PAD_MS=200
FFMPEG_PRESET={FFMPEG_PRESET}
RENDER_CRF={RENDER_CRF}
RENDER_AUDIO_BITRATE=128k
RENDER_FPS=30
TELEGRAM_MAX_VIDEO_MB=50.0
MAX_CLIP_DURATION_SEC=60.0
MIN_CLIP_DURATION_SEC=15.0
FACE_MODEL={FACE_MODEL}
FACE_SAMPLE_FPS=1.0
FACE_SMOOTH_WINDOW=5
FACE_MAX_PAN_PX_PER_SEC=200.0
SMOOTH_CROP=true
SNAP_TO_SILENCE=true
SNAP_WINDOW_SEC=0.8
LOG_LEVEL=INFO
YT_CLIENT_ID={YT_CLIENT_ID}
YT_CLIENT_SECRET={YT_CLIENT_SECRET}
YT_REFRESH_TOKEN={YT_REFRESH_TOKEN}
META_PAGE_TOKEN={META_PAGE_TOKEN}
FB_PAGE_ID={FB_PAGE_ID}
IG_USER_ID={IG_USER_ID}
YOUTUBE_COOKIES_FILE={cookies}
'''
open('.env', 'w').write(env)
open(os.path.join(CONFIG_DIR, '.env'), 'w').write(env)
print('Wrote .env ✓ (also backed up to', CONFIG_DIR + '/.env)')
print('VIDEO_URL =', VIDEO_URL)
print('MAX_CLIPS =', MAX_CLIPS, '| subs =', SUBTITLES_MODE, '| vertical =', VERTICAL_CROP)
print('USE_LOCAL_VIDEO =', USE_LOCAL_VIDEO, ('-> ' + LOCAL_VIDEO_PATH) if USE_LOCAL_VIDEO else '')
print('SEND_TO_TELEGRAM =', SEND_TO_TELEGRAM, ('(needs TELEGRAM_BOT_TOKEN+CHAT_ID)') if SEND_TO_TELEGRAM else '(files only)')
print('APPROVAL_BUTTONS =', USE_APPROVAL_BUTTONS, '| POLLER =', START_TELEGRAM_POLLER)
print('YT configured:', bool(YT_CLIENT_ID and YT_CLIENT_SECRET and YT_REFRESH_TOKEN), '| Meta IG:', bool(META_PAGE_TOKEN and IG_USER_ID), '| Meta FB:', bool(META_PAGE_TOKEN and FB_PAGE_ID))

## 5. Point `storage/` at `/kaggle/working` (must run BEFORE importing `app.*`)

`app/config.py` creates `storage/` dirs at import time, so the symlink has to exist first. `/kaggle/working` persists across sessions (with Files-only persistence); the container filesystem does not.

In [ ]:
import os, shutil
os.chdir('/kaggle/working/autoclips-python')
assert os.path.exists('.env'), 'No .env — run cell 4 first!'

STORE = '/kaggle/working/autoclips-storage'
for d in ['downloads', 'clips', 'tmp', 'videos', 'logs']:
    os.makedirs(f'{STORE}/{d}', exist_ok=True)

if os.path.islink('storage'):
    print('storage/ already linked ->', os.readlink('storage'))
    if os.readlink('storage') != STORE:
        os.unlink('storage')
        shutil.rmtree('storage', ignore_errors=True)
        os.symlink(STORE, 'storage')
        print('re-pointed storage/ ->', STORE)
else:
    shutil.rmtree('storage', ignore_errors=True)
    os.symlink(STORE, 'storage')
    print('storage/ ->', os.readlink('storage'))

## 6. Verify settings (no server on Kaggle)

There is nothing listening on port 8000 here — outbound only (yt-dlp, Gemini). This cell confirms the `.env` + GPU settings the batch cells will actually use.

In [ ]:
import os
os.chdir('/kaggle/working/autoclips-python')
from app.config import settings, CLIPS_DIR, DOWNLOAD_DIR
print('whisper:', settings.WHISPER_MODEL_SIZE, settings.WHISPER_DEVICE, settings.WHISPER_COMPUTE_TYPE)
print('encode: preset=%s crf=%s audio=%s fps=%s' % (settings.FFMPEG_PRESET, settings.RENDER_CRF, settings.RENDER_AUDIO_BITRATE, settings.RENDER_FPS))
print('caps: telegram=%.1fMB max_clip=%.0fs min_clip=%.0fs' % (settings.TELEGRAM_MAX_VIDEO_MB, settings.MAX_CLIP_DURATION_SEC, settings.MIN_CLIP_DURATION_SEC))
print('face:', settings.FACE_MODEL, '| snap:', settings.SNAP_TO_SILENCE, settings.SNAP_WINDOW_SEC)
print('cookies:', repr(settings.YOUTUBE_COOKIES_FILE or '(none)'))
print('clips dir:', CLIPS_DIR, '| downloads dir:', DOWNLOAD_DIR)
import shutil
assert shutil.which('ffmpeg') and shutil.which('ffprobe'), 'ffmpeg/ffprobe missing — re-run cell 2'
print('ffmpeg OK ✓  (no uvicorn on Kaggle — batch mode uses direct function calls)')

## 7. Analyze only (no rendering) — review Gemini's picks first

Downloads (cached by `video_id` under `storage/downloads/`) → native transcript → Gemini candidates. Re-running with the same URL reuses the download + transcript cache.

In [ ]:
import json, os
from pathlib import Path
os.chdir('/kaggle/working/autoclips-python')

# VIDEO_URL / MAX_CLIPS come from cell 4 — override here per-run if you like.
assert 'VIDEO_ID' not in VIDEO_URL, 'Set a real VIDEO_URL in cell 4 first!'
assert GEMINI_API_KEY and not GEMINI_API_KEY.startswith('your-'), 'Set GEMINI_API_KEY in cell 4 first!'

if USE_LOCAL_VIDEO:
    from app.services.video.ids import get_video_id  # noqa: F401  (kept for parity)
    video_path = Path(LOCAL_VIDEO_PATH)
    assert video_path.exists(), f'Local video not found: {video_path}'
    video_id = video_path.stem
    print('Using local video:', video_path)
else:
    from app.services.viral import prepare
    video_id, video_path = prepare(VIDEO_URL)
    print('Downloaded/cached:', video_path)

from app.services.viral import analyze_viral_clips
candidates = analyze_viral_clips(video_id, video_path, MAX_CLIPS)
print(f'\nGemini found {len(candidates)} clip(s):')
for i, c in enumerate(candidates, 1):
    print(f"{i}. [{c.get('start')} -> {c.get('end')}] score={c.get('score')} {c.get('title')}")
    print(f"   reason: {c.get('reason')}")
    print(f"   hashtags: {' '.join('#'+h for h in (c.get('hashtags') or []))}")
# Full JSON for copy/paste
print(json.dumps(candidates, indent=1, ensure_ascii=False)[:4000])

## 8. Render ALL clips (vertical + burned-in subs) → `/kaggle/working`

Renders every candidate from cell 7 (or re-analyzes if you skipped it). Files land in `storage/clips/` (= `/kaggle/working/autoclips-storage/clips/`) and preview inline. Long videos take a while — each clip prints a heartbeat line so the idle timeout doesn't kill the session.

In [ ]:
import os, uuid
from pathlib import Path
os.chdir('/kaggle/working/autoclips-python')
assert 'VIDEO_ID' not in VIDEO_URL, 'Set a real VIDEO_URL in cell 4 first!'

from IPython.display import Video, display
from app.config import CLIPS_DIR
from app.services.video.render import render_video
from app.services.video.timeutils import time_to_seconds
from app.services.viral import analyze_viral_clips, prepare, resolve_subtitle_words

if USE_LOCAL_VIDEO:
    video_path = Path(LOCAL_VIDEO_PATH)
    assert video_path.exists(), f'Local video not found: {video_path}'
    video_id = video_path.stem
else:
    video_id, video_path = prepare(VIDEO_URL)
print('source:', video_path)

try:
    candidates
    print(f'Reusing {len(candidates)} candidates from cell 7 ✓')
except NameError:
    candidates = analyze_viral_clips(video_id, video_path, MAX_CLIPS)
    print(f'Analyzed fresh: {len(candidates)} candidate(s)')

subtitle_words = resolve_subtitle_words(video_id, video_path, SUBTITLES_MODE) if SUBTITLES_MODE != 'none' else None
print('subtitle words:', len(subtitle_words) if subtitle_words else 0, f'({SUBTITLES_MODE})')

rendered = []
for i, c in enumerate(candidates, 1):
    try:
        ss, se = time_to_seconds(c['start']), time_to_seconds(c['end'])
    except (KeyError, ValueError, TypeError) as e:
        print(f'— clip {i}: skipping malformed timestamps ({e})')
        continue
    if se <= ss:
        print(f'— clip {i}: skipping end<=start')
        continue
    out = CLIPS_DIR / f'{video_id}-kaggle-{i}-{uuid.uuid4().hex[:8]}.mp4'
    print(f'[{i}/{len(candidates)}] rendering "{c.get("title")}" [{c.get("start")} -> {c.get("end")}] ...', flush=True)
    try:
        render_video(video_path, out, ss, se, words=subtitle_words, vertical_crop=VERTICAL_CROP)
    except Exception as e:
        print(f'— clip {i}: render FAILED: {e}')
        continue
    size_mb = out.stat().st_size / (1024 * 1024)
    print(f'  ✓ {out.name} ({size_mb:.1f} MB)')
    rendered.append((out, c))

print(f'\nDone: {len(rendered)}/{len(candidates)} rendered -> {CLIPS_DIR}')
for out, c in rendered:
    print(f'\n{c.get("title")} [{c.get("start")} -> {c.get("end")}] score={c.get("score")}')
    display(Video(str(out), embed=True, width=360))

## 8b. Send rendered clips to Telegram (with Approve/Deny buttons)

Needs `TELEGRAM_BOT_TOKEN` + `TELEGRAM_CHAT_ID` + `SEND_TO_TELEGRAM=True` in cell 4. Reuses the repo's `send_clip_to_telegram()` when `USE_APPROVAL_BUTTONS=True`, so each clip carries ✅/❌ (+ IG/FB when configured) exactly like local/Colab. Button **taps** are handled by the poller in cell 8c — run it before tapping. Oversize clips (>50 MB) are skipped with a warning and kept in `storage/clips/`.


In [ ]:
import asyncio, os
os.chdir('/kaggle/working/autoclips-python')

if not SEND_TO_TELEGRAM:
    print('SEND_TO_TELEGRAM=False — skipping (set it True in cell 4 to send). Files stay in storage/clips/.')
else:
    assert TELEGRAM_BOT_TOKEN and not str(TELEGRAM_BOT_TOKEN).startswith('123'), 'Set TELEGRAM_BOT_TOKEN in cell 4 first!'
    assert TELEGRAM_CHAT_ID, 'Set TELEGRAM_CHAT_ID in cell 4 first!'
    # Sync notebook vars -> app settings (settings was imported in cell 6 from .env,
    # but re-runs of cell 4 need a refresh without a kernel restart).
    from app.config import settings as _s
    _s.TELEGRAM_BOT_TOKEN = TELEGRAM_BOT_TOKEN
    _s.TELEGRAM_CHAT_ID = str(TELEGRAM_CHAT_ID)
    _s.YT_CLIENT_ID, _s.YT_CLIENT_SECRET, _s.YT_REFRESH_TOKEN = YT_CLIENT_ID, YT_CLIENT_SECRET, YT_REFRESH_TOKEN
    _s.META_PAGE_TOKEN, _s.FB_PAGE_ID, _s.IG_USER_ID = META_PAGE_TOKEN, FB_PAGE_ID, IG_USER_ID
    try:
        rendered
        clips = [(o, c) for o, c in rendered if os.path.exists(o)]
        print(f'Using {len(clips)} clip(s) rendered in cell 8 ✓')
    except NameError:
        from app.config import CLIPS_DIR as _CD
        from pathlib import Path as _P
        clips = [(_P(p), {'title': _P(p).stem}) for p in sorted(_P(_CD).glob('*-kaggle-*.mp4'))]
        print(f'Cell 8 not run in this session — found {len(clips)} file(s) in storage/clips/')
    assert clips, 'Nothing to send — run cell 8 first.'
    print('sender:', 'send_clip_to_telegram (Approve/Deny buttons)' if USE_APPROVAL_BUTTONS else 'raw Bot.send_video (no buttons)')

    if USE_APPROVAL_BUTTONS:
        from app.telegram.sending import send_clip_to_telegram
        from app.services.video.timeutils import seconds_to_time as _s2t
        from app.services.video.timeutils import time_to_seconds as _t2s
        async def _send_all():
            sent, failed = 0, 0
            for i, (out, c) in enumerate(clips, 1):
                c = c if isinstance(c, dict) else {}
                try:
                    _ss, _se = _t2s(c.get('start', '0')), _t2s(c.get('end', '0'))
                    _dur = round(_se - _ss, 1)
                except Exception:
                    _dur = 0.0
                print(f"— [{i}/{len(clips)}] sending {out.name} with Approve/Deny ...", flush=True)
                try:
                    ok = await send_clip_to_telegram(video_path=out, clip_id=f'{out.stem}', title=c.get('title') or out.stem, score=c.get('score'), reason=c.get('reason'), hashtags=c.get('hashtags') or [], description=c.get('description'), source_video_id=c.get('_source_video_id') or c.get('source_video_id') or '', credit=c.get('_credit') or c.get('credit') or '')
                except Exception as e:
                    print(f'  ✗ FAILED: {type(e).__name__}: {e}')
                    failed += 1
                    continue
                print('  ✓ sent (buttons attached)' if ok else '  ⚠ sent/kept without buttons (see log — oversize or not delivered)')
                sent += 1 if ok else 0
                failed += 0 if ok else 1
            return sent, failed
        sent, failed = asyncio.run(_send_all())
        print(f'\nTelegram: {sent} delivered with buttons, {failed} failed/skipped.')
        if START_TELEGRAM_POLLER:
            print('Next: run cell 8c to start the poller BEFORE tapping Approve/Deny.')
    else:
        from telegram import Bot
        from app.config import settings
        bot = Bot(token=TELEGRAM_BOT_TOKEN)
        cap_mb = float(getattr(settings, 'TELEGRAM_MAX_VIDEO_MB', 50.0) or 50.0)
        async def _send_all():
            sent, skipped = 0, 0
            for i, (out, c) in enumerate(clips, 1):
                size_mb = os.path.getsize(out) / (1024 * 1024)
                title = (c.get('title') if isinstance(c, dict) else None) or out.name
                score = (c.get('score') if isinstance(c, dict) else None) or '?'
                reason = (c.get('reason') if isinstance(c, dict) else None) or ''
                tags = (c.get('hashtags') if isinstance(c, dict) else None) or []
                desc = (c.get('description') if isinstance(c, dict) else None) or ''
                tagline = ' '.join('#' + str(t).replace(' ', '') for t in tags[:6])
                caption = f'\U0001F3AC {title}\n\U0001F525 Score: {score} | clip {i}/{len(clips)}'
                if tagline:
                    caption += f'\n{tagline}'
                if desc:
                    caption += f'\n\n{desc[:700]}'
                if reason:
                    caption += f'\n\n\U0001F4A1 {reason[:300]}'
                if len(caption) > 1024:
                    caption = caption[:1000] + '\n…(cont.)'
                if size_mb > cap_mb:
                    print(f'— [{i}/{len(clips)}] SKIP {out.name} ({size_mb:.1f} MB > {cap_mb:.0f} MB cap)')
                    skipped += 1
                    continue
                print(f'— [{i}/{len(clips)}] sending {out.name} ({size_mb:.1f} MB) ...', flush=True)
                try:
                    with open(out, 'rb') as f:
                        await bot.send_video(chat_id=TELEGRAM_CHAT_ID, video=f, caption=caption)
                    print('  ✓ sent')
                    sent += 1
                except Exception as e:
                    print(f'  ✗ FAILED: {type(e).__name__}: {e}')
                    skipped += 1
            return sent, skipped
        sent, skipped = asyncio.run(_send_all())
        print(f'\nTelegram: {sent} sent, {skipped} skipped.')


## 8c. Start Telegram poller (makes Approve/Deny taps work)

Same handlers as local `uvicorn app.main:app` lifespan (`/start`, buttons, text router) but **without any server** — outbound long-polling only, so it works on Kaggle. Run once per session **before tapping buttons**. Stop all other pollers first (local/Colab) or Telegram returns `Conflict: terminated by other getUpdates`. In-memory `pending_uploads`/`jobs` die with the kernel — files in `/kaggle/working` survive.


In [ ]:
import asyncio, os, threading
os.chdir('/kaggle/working/autoclips-python')

if not SEND_TO_TELEGRAM:
    print('SEND_TO_TELEGRAM=False — poller not started (nothing was sent). Set True in cell 4 if you want buttons.')
elif not START_TELEGRAM_POLLER:
    print('START_TELEGRAM_POLLER=False — skipping (buttons were sent but taps will be ignored until you run this with True).')
else:
    assert TELEGRAM_BOT_TOKEN, 'Set TELEGRAM_BOT_TOKEN in cell 4 first!'
    from app.config import settings as _s2
    _s2.TELEGRAM_BOT_TOKEN = TELEGRAM_BOT_TOKEN
    _s2.TELEGRAM_CHAT_ID = str(TELEGRAM_CHAT_ID)
    _s2.YT_CLIENT_ID, _s2.YT_CLIENT_SECRET, _s2.YT_REFRESH_TOKEN = YT_CLIENT_ID, YT_CLIENT_SECRET, YT_REFRESH_TOKEN
    _s2.META_PAGE_TOKEN, _s2.FB_PAGE_ID, _s2.IG_USER_ID = META_PAGE_TOKEN, FB_PAGE_ID, IG_USER_ID
    if not (_s2.YT_CLIENT_ID and _s2.YT_CLIENT_SECRET and _s2.YT_REFRESH_TOKEN):
        print('⚠ YT_* missing — ✅ YouTube button will fail until you paste YT_CLIENT_ID/SECRET/REFRESH_TOKEN in cell 4.')
    if not (_s2.META_PAGE_TOKEN and _s2.IG_USER_ID):
        print('ℹ IG not configured — 📸 button hidden (paste META_PAGE_TOKEN/IG_USER_ID to enable).')
    if not (_s2.META_PAGE_TOKEN and _s2.FB_PAGE_ID):
        print('ℹ FB not configured — 📘 button hidden (paste META_PAGE_TOKEN/FB_PAGE_ID to enable).')
    _flag = '_KAGGLE_POLLER_STARTED'
    import builtins
    if getattr(builtins, _flag, False):
        print('Poller already running in this kernel ✓ — tap buttons in Telegram.')
    else:
        from telegram.ext import CallbackQueryHandler, CommandHandler, MessageHandler, filters
        from app.telegram.bot import _polling_error_callback, get_app
        from app.telegram.handlers_buttons import telegram_button
        from app.telegram.handlers_start import telegram_start, telegram_text_router
        def _run_poller():
            async def _main():
                app = get_app()  # reuses repo Bot/Application builders
                # same 3 handlers as app/main.py lifespan (no duplicates on re-run)
                for h in list(app.handlers.get(0, [])):
                    try:
                        app.remove_handler(h)
                    except Exception:
                        pass
                app.add_handler(CommandHandler('start', telegram_start))
                app.add_handler(CallbackQueryHandler(telegram_button))
                app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, telegram_text_router))
                await app.initialize()
                await app.start()
                await app.updater.start_polling(poll_interval=2.0, timeout=30, bootstrap_retries=-1, drop_pending_updates=False, error_callback=_polling_error_callback)
                print('Telegram poller UP ✓ — Approve/Deny taps now work. Keep this kernel alive.')
                await asyncio.Event().wait()  # run until kernel dies
            asyncio.run(_main())
        _t = threading.Thread(target=_run_poller, daemon=True, name='kaggle-tg-poller')
        _t.start()
        setattr(builtins, _flag, True)
        print('Poller thread started in background. Tap a button in Telegram to test.')


## 9. Files, logs, quota

`/kaggle/working` caps at ~20GB — download MP4s you want to keep, then clean `downloads/`/`tmp/` if needed. Right-click files in the Kaggle file browser to download.

In [ ]:
import os
os.chdir('/kaggle/working/autoclips-python')
!echo '--- clips ---' && ls -lh storage/clips/ 2>/dev/null | tail -20
!echo '--- disk ---' && du -sh /kaggle/working/* 2>/dev/null
!echo '--- app.log (last 30) ---' && tail -n 30 storage/logs/app.log 2>/dev/null || echo '(no log yet)'

## Notes / troubleshooting (Kaggle-specific)

- **Settings checklist:** Accelerator **GPU T4 x2**, Internet **ON**, Persistence **Files only**. `Commit` (batch run) disables Internet — use interactive sessions for YouTube/Gemini.
- **No server here:** `uvicorn` + Telegram polling from `colab_setup.ipynb` cell 7 is intentionally absent — Kaggle blocks inbound ports. Batch cells 7–8 replace `/clips/analyze` + `/clips/viral`.
- **YouTube bot-check (`Sign in to confirm you're not a bot`):** Kaggle IPs trigger this often. Fix: export `cookies.txt` from your logged-in browser (after accepting the video's consent page), upload to `/kaggle/working/cookies.txt`, re-run cells 4–8. Or set `USE_LOCAL_VIDEO=True` and upload the MP4 directly.
- **Out of VRAM:** set `WHISPER_MODEL_SIZE='medium'` in cell 4 and re-run 4–8. `large-v3-turbo` is the T4 sweet spot but tight alongside YOLO face tracking.
- **Session lost?** `/kaggle/working/autoclips-storage` (downloads/clips/transcripts) + `/kaggle/working/autoclips-config/.env` survive restarts — re-run from cell 4 (paste values again) or just 7–8 if files are intact.
- **Idle timeout:** cells print a line per clip; for very long transcribes, watch `storage/logs/app.log` (cell 9). `Commit/Save` runs won't have Internet — stay interactive.
- **Telegram:** cell 8b reuses the repo's `send_clip_to_telegram()` (Approve/Deny + YT/IG/FB buttons); cell 8c runs the repo's polling handlers without any server. Stop other pollers (local/Colab) first or Telegram `Conflict`s. `pending_uploads`/`jobs` are in-memory — a kernel restart expires buttons, files in `/kaggle/working` stay.